In [ ]:
import torch
import matplotlib.pyplot as plt
N= 2000
x= torch.linspace(-0.01, 0.01, N, dtype=torch.float64)
y= torch.linspace(-0.01, 0.01, N, dtype= torch.float64)
X, Y = torch.meshgrid(x, y, indexing='xy')
amp= torch.exp(-(X**2 + Y**2)/0.0003**2)
ph = X
field= amp*torch.exp(1j*ph)
plt.imshow(torch.abs(field).numpy())
plt.show()

def propagate(field, wavelength, z, dx, dy):
    N= field.shape[0]
    fx= torch.fft.fftfreq(N, d= dx, dtype= torch.float64)
    fy= torch.fft.fftfreq(N, d= dy, dtype= torch.float64)
    FX, FY= torch.meshgrid(fx, fy, indexing= 'xy')
    arg= (2*torch.pi/wavelength)**2 - (2*torch.pi*FX)**2 - (2*torch.pi*FY)**2
    kz= torch.sqrt(arg.to(torch.complex128))
    H= torch.exp(1j*kz*z)

    field_freq= torch.fft.fft2(field)
    propagated_freq= field_freq* H
    propagated_field= torch.fft.ifft2(propagated_freq)
    return(propagated_field)

propagated1= propagate(field, wavelength= 633e-9, z= 0.0001, dx= x[1] - x[0], dy= y[1]- y[0])
intensity1= torch.abs(propagated1)**2
# print(intensity1)    # printing same intensities as that of numpy

R= torch.sqrt(X**2 +Y**2)
r= 0.0006
a= ((R<r).to(torch.float64))
field= a
propagated2= propagate(a, wavelength= 633e-9, z=1, dx=x[1]-x[0], dy=y[1]- y[0])
intensity2= torch.abs(propagated2)**2
# plt.imshow(intensity2)
# plt.show()
wavelength, z = 633e-9, 1
k= 2*torch.pi/wavelength
u = k*r*(R/z)
analytic = torch.where(u==0, 1.0, 2* torch.special.bessel_j1(u)/u )**2
sim_norm = intensity2/intensity2.max()
analy_norm = analytic/analytic.max()
mid = N//2
plt.plot(x, sim_norm[mid,:], label='simulated')
plt.plot(x, analy_norm[mid,:], label='analytic Airy')
plt.legend()
plt.xlabel('x (m)')
plt.ylabel('normalized intensity')
plt.title('Simulated vs Analytic Airy Pattern')
plt.show()

half_width = 0.0001
slit = ((torch.abs(X)< half_width).to(torch.float64))
field = slit
propagated3 = propagate(slit, wavelength=633e-9, z=0.5, dx=x[1]-x[0], dy=y[1]-y[0])
intensity3 = torch.abs(propagated3)**2
plt.imshow(intensity3)
plt.show()
wavelength, z = 633e-9, 0.5
d= 2*half_width
sin= X/z
x_arg= d*sin/wavelength
anal_slit = torch.sinc(x_arg)**2
sim_norm = intensity3/intensity3.max()
analy_norm = anal_slit/anal_slit.max()
mid = N//2
plt.plot(x, sim_norm[mid,:], label='simulated')
plt.plot(x, analy_norm[mid,:], label='analytic Slit')
plt.legend()
plt.xlabel('x (m)')
plt.ylabel('normalized intensity')
plt.title('Simulated vs Analytic single slit diffraction')
plt.show()

x0, w0= 0.002, 0.0005
input_A= torch.exp(-(((X-(-x0))**2+Y**2))/w0**2)
input_B = torch.exp(-(((X-x0)**2+Y**2))/w0**2)

region1 = (Y > 0).to(torch.float64)   # top half — input_A should land here
region2 = (Y < 0).to(torch.float64)   # bottom half — input_B should land here

mask = torch.nn.Parameter(torch.zeros(N, N, dtype=torch.float64))

def loss_fn(mask):
    fA = input_A * torch.exp(1j*mask)
    pA = propagate(fA, wavelength=633e-9, z=1, dx=x[1]-x[0], dy=y[1]-y[0])
    iA = torch.abs(pA)**2
    wrongA = torch.sum(iA*region2) / torch.sum(iA)

    fB = input_B * torch.exp(1j*mask)
    pB = propagate(fB, wavelength=633e-9, z=1, dx=x[1]-x[0], dy=y[1]-y[0])
    iB = torch.abs(pB)**2
    wrongB = torch.sum(iB*region1) / torch.sum(iB)

    return wrongA + wrongB
print("untrained mask loss:", loss_fn(mask).item())

optimizer= torch.optim.Adam([mask], lr= 0.01)

losses=[]
for step in range(1000):
    optimizer.zero_grad()
    loss= loss_fn(mask)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if step %100==0:
        print(f"step{step}: loss= {loss_fn(mask).item():.6f}")
print(loss_fn(mask).item())
plt.plot(losses)
plt.xlabel("training step")
plt.ylabel("loss")
plt.title("Training loss over time")
plt.show()

plt.imshow(mask.detach().numpy())
plt.colorbar()
plt.title("Trained phase mask")
plt.show()

fA = input_A * torch.exp(1j*mask)
pA = propagate(fA, wavelength=633e-9, z=1, dx=x[1]-x[0], dy=y[1]-y[0])
iA = torch.abs(pA)**2
plt.imshow(iA.detach().numpy(), origin= 'lower')
plt.title("Input A output (should be mostly top)")
plt.show()

fB = input_B * torch.exp(1j*mask)
pB = propagate(fB, wavelength=633e-9, z=1, dx=x[1]-x[0], dy=y[1]-y[0])
iB = torch.abs(pB)**2
plt.imshow(iB.detach().numpy(), origin= 'lower')
plt.title("Input B output (should be mostly bottom)")
plt.show()

print("Input A -> top:", torch.sum(iA*region1).item(), " bottom:", torch.sum(iA*region2).item())
print("Input B -> top:", torch.sum(iB*region1).item(), " bottom:", torch.sum(iB*region2).item())